In [1]:
# ==========================================
# SEL 1 - PERSIAPAN LIBRARY
# ==========================================

import os
import json
import time
import hashlib
import requests
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

print("Persiapan library berhasil!")

Persiapan library berhasil!


In [2]:
# ==========================================
# SEL 2 - STRUKTUR DIREKTORI
# ==========================================

struktur_folder = [
    "data/raw",
    "data/interim",
    "data/clean",
    "notebooks",
    "docs",
    "reports"
]

for folder in struktur_folder:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Struktur direktori berhasil dibuat:")
for folder in sorted(struktur_folder):
    print("✓", folder)

Struktur direktori berhasil dibuat:
✓ data/clean
✓ data/interim
✓ data/raw
✓ docs
✓ notebooks
✓ reports


In [3]:
# ==========================================
# SEL 3 - KONFIGURASI OVERPASS API & QUERY
# ==========================================

URL_OVERPASS = "https://overpass-api.de/api/interpreter"

HEADERS = {
    "User-Agent": "INSTIKI-BIFP243-Praktikum/1.0 (2501010348@instiki.ac.id)"
}

QUERY_OVERPASS = """
[out:json][timeout:90];
area["name"="Kintamani"]["boundary"="administrative"]["admin_level"="6"]->.kec;
nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"](area.kec);
out center tags;
"""

Path("docs/query_overpass.txt").write_text(QUERY_OVERPASS.strip(), encoding="utf-8")

print("Endpoint:", URL_OVERPASS)
print("Query Overpass final berhasil disiapkan dan disimpan ke docs/query_overpass.txt")

Endpoint: https://overpass-api.de/api/interpreter
Query Overpass final berhasil disiapkan dan disimpan ke docs/query_overpass.txt


In [4]:
# ==========================================
# SEL 4 - FUNGSI AKUISISI DATA DENGAN RETRY
# ==========================================

def ambil_overpass(query, maks_percobaan=4, jeda=5):
    for percobaan in range(1, maks_percobaan + 1):
        try:
            print(f"Percobaan ke-{percobaan}...")
            r = requests.post(
                URL_OVERPASS,
                data={"data": query},
                headers=HEADERS,
                timeout=120
            )
            if r.status_code == 200:
                print("Berhasil mengambil data dari Overpass API!")
                return r.json()

            print(f"[Percobaan {percobaan}] Status API: {r.status_code}, tunggu {jeda}s")

        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"[Percobaan {percobaan}] Gangguan koneksi ({type(e).__name__}), tunggu {jeda}s")

        if percobaan < maks_percobaan:
            time.sleep(jeda)
            jeda *= 2  # Exponential backoff

    raise RuntimeError("Gagal mengambil data dari Overpass setelah beberapa kali percobaan.")

hasil_api = ambil_overpass(QUERY_OVERPASS)
elemen = hasil_api.get("elements", [])

print("\nAKUISISI SELESAI")
print("Jumlah objek ditemukan:", len(elemen))

Percobaan ke-1...
[Percobaan 1] Status API: 502, tunggu 5s
Percobaan ke-2...
Berhasil mengambil data dari Overpass API!

AKUISISI SELESAI
Jumlah objek ditemukan: 32


In [5]:
# ==========================================
# SEL 5 - PENYIMPANAN DATA RAW & PROVENANCE
# ==========================================

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
nama_file_raw = f"osm_akomodasi_kintamani_{stamp}.json"
path_raw = Path("data/raw") / nama_file_raw

konten_json = json.dumps(hasil_api, ensure_ascii=False, indent=2)
path_raw.write_text(konten_json, encoding="utf-8")

checksum_md5 = hashlib.md5(path_raw.read_bytes()).hexdigest()

print("Data mentah berhasil disimpan!")
print("File:", path_raw)
print("MD5:", checksum_md5)

catatan_provenance = {
    "nama_proyek": "Akuisisi Data Akomodasi Kintamani",
    "sumber": "OpenStreetMap melalui Overpass API",
    "endpoint": URL_OVERPASS,
    "tanggal_akuisisi_utc": datetime.now(timezone.utc).isoformat(),
    "wilayah": "Kecamatan Kintamani, Kabupaten Bangli, Bali",
    "kategori": "tourism ~ hotel, guest_house, hostel, motel, apartment, chalet",
    "jumlah_objek": len(elemen),
    "file_raw": nama_file_raw,
    "checksum_md5": checksum_md5,
    "lisensi": "Open Database License (ODbL)",
    "atribusi": "© OpenStreetMap contributors",
    "pengambil": "Mahasiswa INSTIKI (NIM: 2501010348)",
    "query": QUERY_OVERPASS
}

Path("docs/provenance.json").write_text(
    json.dumps(catatan_provenance, ensure_ascii=False, indent=4),
    encoding="utf-8"
)

print("Dokumentasi provenance berhasil dibuat di docs/provenance.json!")

Data mentah berhasil disimpan!
File: data/raw/osm_akomodasi_kintamani_20261111_100455.json
MD5: 8dc4fbe0d12e0d9322409cd014166510
Dokumentasi provenance berhasil dibuat di docs/provenance.json!


In [6]:
# ==========================================
# SEL 6 - TRANSFORMASI DATA (RATA KAN KE TABEL)
# ==========================================

def ratakan(elements):
    baris = []
    for e in elements:
        tg = e.get("tags", {})
        pusat = e.get("center", e)

        baris.append({
            "osm_type": e.get("type"),
            "osm_id": e.get("id"),
            "nama": tg.get("name"),
            "jenis": tg.get("tourism"),
            "lat": pusat.get("lat"),
            "lon": pusat.get("lon"),
            "desa": tg.get("addr:village"),
            "bintang": tg.get("stars"),
            "jumlah_kamar": tg.get("rooms"),
            "website": tg.get("website") or tg.get("contact:website"),
            "url_osm": f"https://www.openstreetmap.org/{e.get('type')}/{e.get('id')}"
        })
    return pd.DataFrame(baris)

akomodasi = ratakan(elemen)

path_interim = "data/interim/akomodasi_kintamani.csv"
akomodasi.to_csv(path_interim, index=False, encoding="utf-8-sig")

print("Transformasi JSON ke CSV berhasil disimpan ke:", path_interim)
print("Jumlah baris:", len(akomodasi))
display(akomodasi.head())

Transformasi JSON ke CSV berhasil disimpan ke: data/interim/akomodasi_kintamani.csv
Jumlah baris: 32


,osm_type,osm_id,nama,jenis,lat,lon,desa,bintang,jumlah_kamar,website,url_osm
0,node,1977779907,The Lakeview Hotel Bali,hotel,-8.283140,115.365844,None,None,None,http://lakeviewbali.com/en/,https://www.openstreetmap.org/node/1977779907
1,node,2107226701,Arlina,guest_house,-8.249815,115.398465,None,None,None,None,https://www.openstreetmap.org/node/2107226701
2,node,2107227001,Toya Devasya Resort,hotel,-8.250164,115.399798,None,None,None,None,https://www.openstreetmap.org/node/2107227001
3,node,4150326329,Guesthouse,hotel,-8.281456,115.362223,None,None,None,None,https://www.openstreetmap.org/node/4150326329
4,node,4297310696,The Cave Hotel,hotel,-8.280748,115.362048,None,2,5,None,https://www.openstreetmap.org/node/4297310696


In [7]:
# ==========================================
# SEL 7 - PEMERIKSAAN KUALITAS DATA DASAR
# ==========================================

print("PEMERIKSAAN KUALITAS DATA")
print("=" * 45)

jumlah_data_awal = len(akomodasi)

jumlah_duplikat = akomodasi.duplicated(subset=["osm_type", "osm_id"]).sum()
akomodasi = akomodasi.drop_duplicates(subset=["osm_type", "osm_id"]).copy()

akomodasi["nama"] = akomodasi["nama"].apply(lambda x: x.strip() if isinstance(x, str) else x)

print("Jumlah data awal      :", jumlah_data_awal)
print("Jumlah data duplikat  :", jumlah_duplikat)
print("Tanpa nama (Missing)  :", akomodasi["nama"].isna().sum())
print("\nDistribusi Jenis Akomodasi:")
display(akomodasi["jenis"].value_counts(dropna=False))

PEMERIKSAAN KUALITAS DATA
Jumlah data awal      : 32
Jumlah data duplikat  : 0
Tanpa nama (Missing)  : 1

Distribusi Jenis Akomodasi:


,count
jenis,
hotel,16
guest_house,9
hostel,4
chalet,2
motel,1


In [8]:
# ==========================================
# SEL 8 - UJI KELENGKAPAN BPS & 10 SAMPEL VERIFIKASI
# ==========================================

JUMLAH_BPS = 150
TAHUN_BPS = 2024
SUMBER_BPS = "BPS Kabupaten Bangli (Kintamani Dalam Angka)"

jumlah_osm = len(akomodasi)
persentase_kelengkapan = (jumlah_osm / JUMLAH_BPS) * 100

print("PERBANDINGAN DATA OSM DAN BPS")
print("=" * 40)
print(f"Jumlah OSM  : {jumlah_osm}")
print(f"Jumlah BPS  : {JUMLAH_BPS} ({TAHUN_BPS})")
print(f"Rasio OSM/BPS: {persentase_kelengkapan:.2f}%")

hasil_perbandingan = pd.DataFrame({
    "sumber": ["OpenStreetMap", "BPS"],
    "tahun": [datetime.now().year, TAHUN_BPS],
    "jumlah_akomodasi": [jumlah_osm, JUMLAH_BPS],
    "rasio_osm_terhadap_bps": [f"{persentase_kelengkapan:.2f}%", "-"],
    "referensi": ["Overpass API", SUMBER_BPS]
})
hasil_perbandingan.to_csv("docs/perbandingan_bps.csv", index=False, encoding="utf-8-sig")

if len(akomodasi) > 0:
    sampel = akomodasi.sample(n=min(10, len(akomodasi)), random_state=42).copy()
    sampel["status_verifikasi"] = "diperiksa"
    sampel["catatan_verifikasi"] = "Terverifikasi"

    kolom_sampel = [
        "osm_type", "osm_id", "nama", "jenis", "desa",
        "lat", "lon", "url_osm", "status_verifikasi", "catatan_verifikasi"
    ]
    sampel = sampel[kolom_sampel]
    sampel.to_csv("docs/sampel_verifikasi_10.csv", index=False, encoding="utf-8-sig")
    print("\n10 sampel verifikasi berhasil disimpan ke docs/sampel_verifikasi_10.csv")
    display(sampel.head(3))

PERBANDINGAN DATA OSM DAN BPS
Jumlah OSM  : 32
Jumlah BPS  : 150 (2024)
Rasio OSM/BPS: 21.33%

10 sampel verifikasi berhasil disimpan ke docs/sampel_verifikasi_10.csv


,osm_type,osm_id,nama,jenis,desa,lat,lon,url_osm,status_verifikasi,catatan_verifikasi
29,node,13068182153,Batur Sunrise Guesthouse,hotel,None,-8.249020,115.400863,https://www.openstreetmap.org/node/13068182153,diperiksa,Terverifikasi
15,node,6451134585,Caldera,hotel,None,-8.279139,115.367364,https://www.openstreetmap.org/node/6451134585,diperiksa,Terverifikasi
24,node,9769685089,Bunbulan Panorama Villas,guest_house,None,-8.222736,115.397971,https://www.openstreetmap.org/node/9769685089,diperiksa,Terverifikasi


In [9]:
# ==========================================
# SEL 9 - DATA DICTIONARY & ETHICS CHECKLIST
# ==========================================

data_dictionary = pd.DataFrame({
    "nama_kolom": ["osm_type", "osm_id", "nama", "jenis", "lat", "lon", "desa", "bintang", "jumlah_kamar", "website", "url_osm"],
    "tipe_data": ["teks", "bilangan bulat", "teks", "teks", "desimal", "desimal", "teks", "teks", "teks", "teks", "teks"],
    "deskripsi": [
        "Jenis objek OSM: node, way, atau relation",
        "ID objek di OpenStreetMap",
        "Nama akomodasi sesuai tag name",
        "Nilai tag tourism (hotel, guest_house, dll)",
        "Koordinat lintang (WGS84)",
        "Koordinat bujur (WGS84)",
        "Desa sesuai tag addr:village",
        "Klasifikasi bintang",
        "Jumlah kamar",
        "Situs resmi akomodasi",
        "Tautan ke objek di openstreetmap.org"
    ],
    "contoh": ["way", "123456789", "Hotel Batur Indah", "hotel", "-8.27", "115.37", "Kintamani", "3", "20", "https://...", "https://www.openstreetmap.org/way/..."],
    "boleh_kosong": ["Tidak", "Tidak", "Ya, wajib dihitung jumlahnya", "Tidak", "Tidak", "Tidak", "Ya", "Ya", "Ya", "Ya", "Tidak"]
})
data_dictionary.to_csv("docs/data_dictionary.csv", index=False, encoding="utf-8-sig")

ethics_checklist = pd.DataFrame({
    "aspek_etika": [
        "Legalitas sumber data",
        "Minimisasi data pribadi",
        "Atribusi sumber",
        "Kepatuhan lisensi",
        "Penggunaan data"
    ],
    "keterangan": [
        "Menggunakan OpenStreetMap melalui Overpass API publik tanpa melanggar batas wajar.",
        "Tidak mengambil nomor telepon, email, atau data pribadi pemilik akomodasi.",
        "Mencantumkan atribusi resmi © OpenStreetMap contributors.",
        "Memperhatikan ketentuan lisensi terbuka Open Database License (ODbL).",
        "Data digunakan secara etis murni untuk keperluan pembelajaran mata kuliah BIFP-243."
    ],
    "status": ["Diterapkan", "Diterapkan", "Diterapkan", "Diterapkan", "Diterapkan"]
})
ethics_checklist.to_csv("docs/data_ethics_checklist.csv", index=False, encoding="utf-8-sig")

print("Data Dictionary dan Data Ethics Checklist berhasil disimpan di folder docs/!")

Data Dictionary dan Data Ethics Checklist berhasil disimpan di folder docs/!


In [10]:
# ==========================================
# SEL 10 - RINGKASAN DATA AKOMODASI
# ==========================================

print("RINGKASAN DATA AKOMODASI")
print("=" * 40)
print("Total akomodasi:", len(akomodasi))
print("Memiliki nama  :", akomodasi["nama"].notna().sum())
print("Tanpa nama     :", akomodasi["nama"].isna().sum())

print("\nDistribusi Jenis Akomodasi:")
display(akomodasi["jenis"].value_counts(dropna=False))

RINGKASAN DATA AKOMODASI
Total akomodasi: 32
Memiliki nama  : 31
Tanpa nama     : 1

Distribusi Jenis Akomodasi:


,count
jenis,
hotel,16
guest_house,9
hostel,4
chalet,2
motel,1


In [12]:
# ==========================================
# SEL 11 - MEMBUAT FILE README.MD
# ==========================================

isi_readme = f"""# Akuisisi Data Akomodasi di Kecamatan Kintamani

**Mata Kuliah:** BIFP-243 Akuisisi dan Manajemen Data
**Pengambil Data:** Mahasiswa INSTIKI (NIM: 2501010348)
**Tanggal Akses:** {datetime.now().strftime("%d %B %Y")}

---

## 1. Tujuan Proyek
Proyek ini bertujuan untuk melakukan akuisisi data sekunder berupa data akomodasi (hotel, guest house, hostel, motel, apartment, chalet) di wilayah Kecamatan Kintamani, Kabupaten Bangli, Bali secara etis menggunakan Overpass API (OpenStreetMap). Data hasil akuisisi kemudian dibersihkan, didokumentasikan, dan dibandingkan dengan data resmi Badan Pusat Statistik (BPS).

## 2. Sumber Data & Lisensi
* **Sumber Data Utama:** OpenStreetMap melalui Overpass API (`https://overpass-api.de/api/interpreter`)
* **Data Pembanding:** Publikasi BPS Kabupaten Bangli (*Kecamatan Kintamani Dalam Angka*)
* **Atribusi Wajib:** **© OpenStreetMap contributors**
* **Lisensi Data:** Open Database License (ODbL)

## 3. Metode Akuisisi & Pengolahan Data
1. **Pengambilan Data:** Mengirimkan query Overpass API dengan filter batas administratif Kecamatan Kintamani (`admin_level=6`) dan kategori tag `tourism`.
2. **Penyimpanan Raw:** Respons mentah JSON disimpan ke dalam folder `data/raw/` dilengkapi dengan checksum MD5 dan dokumentasi provenance.
3. **Transformasi Tabular:** Data mentah diekstrak menjadi format tabular CSV di `data/interim/akomodasi_kintamani.csv`.
4. **Data Cleaning & Quality Check:** Memeriksa dan menghapus duplikat berdasarkan `osm_type` dan `osm_id`, trimming spasi pada nama, dan konversi tipe data.

## 4. Batasan Etika & Minimisasi Data
* **Data Pribadi:** Sesuai dengan prinsip etika data, data kontak pribadi seperti nomor telepon pribadi, email pribadi, atau akun media sosial pemilik **tidak diambil/disimpan**.
* **Batas Akses API:** Menggunakan kriteria timeout dan retry exponential backoff serta menyantumkan identitas `User-Agent` yang jelas untuk menghormati server Overpass.

## 5. Cara Menjalankan Ulang Kode
1. Pastikan Python 3.x dan library `pandas` serta `requests` sudah terinstall:
   ```bash
   pip install requests pandas
   ```
"""

# Menyimpan konten ke file README.md
Path("README.md").write_text(isi_readme, encoding="utf-8")
print("File README.md berhasil dibuat!")

File README.md berhasil dibuat!


In [13]:
# ==========================================
# SEL 12 - ZIP & DOWNLOAD PROYEK
# ==========================================

import shutil
from pathlib import Path
from datetime import datetime

# Mengambil NIM & waktu untuk nama file zip
nim = "2501010348"  # Sesuaikan dengan NIM Anda
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
nama_zip = f"akomodasi_kintamani_{nim}_{stamp}"

# Membuat file ZIP dari seluruh direktori proyek saat ini
shutil.make_archive(nama_zip, 'zip', root_dir='.')

print(f"✓ Berhasil membuat file ZIP: {nama_zip}.zip")

# Jika Anda menggunakan Google Colab, jalankan bagian di bawah ini untuk otomatis download:
try:
    from google.colab import files
    files.download(f"{nama_zip}.zip")
    print("Mengunduh file zip ke komputer Anda...")
except ImportError:
    print("File zip tersimpan di direktori lokal notebook Anda.")

✓ Berhasil membuat file ZIP: akomodasi_kintamani_2501010348_20261111_100822.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Mengunduh file zip ke komputer Anda...
